## Method Resolution Order

In order to understand constructor behavior, we first need to understand the concept of MRO. Ref
http://python-history.blogspot.com/2010/06/method-resolution-order.html for more details. In a
nutshell, when invoking a method on a object with an object hierarchy, Python has specific rules of
where and in what order it will start looking for this method.

For linear hierarchies, it is as expected, it starts with the most specific, i.e., with the object
on which the call is invoked, and goes all the way upto the mother object.

In [1]:
class Root:
    pass


class LevelOne(Root):
    pass


class LevelTwo(LevelOne):
    pass

In [2]:
LevelTwo.__mro__

(__main__.LevelTwo, __main__.LevelOne, __main__.Root, object)

### Mulitple Inheritance

For multiple inheritances, it starts as usual with the most specific, and then goes through the
classes from left to right, so in the example below it will first look in LevelOneA and then in
LevelOneB and then in Root.

In [3]:
%reset -f

In [4]:
class Root:
    pass


class LevelOneA(Root):
    pass


class LevelOneB(Root):
    pass


class LevelTwo(LevelOneA, LevelOneB):
    pass

In [5]:
LevelTwo.__mro__

(__main__.LevelTwo,
 __main__.LevelOneA,
 __main__.LevelOneB,
 __main__.Root,
 object)

### Polymorphism

For most methods this behaves as expected. For any method call, the Python runtime will try to find
the method up the object hierarchy in the MRO starting with the object type that was instantiated. In the example below, `self` is of type `LevelOne`, so when `self.method_a()` is called inside `Root::method_b()`, it will start from the bottom again and find `method_b()` on `LevelOne`.

In [6]:
%reset -f

In [7]:
class Root:
    def method_a(self):
        print("Root::method_a")

    def method_b(self):
        print("Root::method_b")
        self.method_a()


class LevelOne(Root):
    def method_a(self):
        print("LevelOne::method_a")

In [8]:
lone = LevelOne()

In [9]:
lone.method_b()

Root::method_b
LevelOne::method_a


Below is a clearer demo of polymorphism following the MRO in case of multiple inheritance. In the example below, calling a method on a `LevelTwo` object, MRO will look for the method in `LevelTwo`, then `LevelOneA`, then `LevelOneB`, and finally in `Root`.

In [10]:
%reset -f

In [11]:
class Root:
    pass


class LevelOneA(Root):
    def method_x(self):
        print("LevelOneA::method_x")

    def common_method(self):
        print("LevelOneA::common_method")


class LevelOneB(Root):
    def method_y(self):
        print("LevelOneB::method_y")

    def common_method(self):
        print("LevelOneB::common_method")


class LevelTwo(LevelOneA, LevelOneB):
    pass

In [12]:
ltwo = LevelTwo()

In [13]:
ltwo.common_method()

LevelOneA::common_method


In [14]:
ltwo.method_y()

LevelOneB::method_y


### Double Underscores

But for methods whose names start with double underscores, the behavior is different and unexpected.
It will only look for that method on the object it is currently executing in, i.e., it will not
start from the bottom of the object hierarchy, nor will it go up the hierarchy.

Below is a simple example that it will not go up the hierarchy.


In [15]:
%reset -f

In [16]:
class Root:
    def __method_a(self):
        print("Root::method_a")

    def method_b(self):
        print("Root::method_b")


class LevelOne(Root):
    pass

In [17]:
lone = LevelOne()

In [18]:
# Will go up the object hierarchy
lone.method_b()

Root::method_b


In [19]:
# Will not go up the object hiearchy
try:
    lone.__method_a()
except AttributeError as ae:
    print("ERROR: ", ae)

ERROR:  'LevelOne' object has no attribute '__method_a'


In the example below, when `self.__method_a()` is called from `Root`, it will **not** start from the bottom, it will look for it in the same level.

In [20]:
%reset -f

In [21]:
class Root:
    def __method_a(self):
        print("Root::__method_a")

    def method_b(self):
        print("Root::method_b")
        self.__method_a()


class LevelOne(Root):
    def __method_a(self):
        print("LevelOne::__method_a")

In [22]:
lone = LevelOne()
lone.method_b()

Root::method_b
Root::__method_a


In the example below, calling `ltwo.method_b()` will go upto `LevelOne.method_b()`, which then calls `self.__method_a()`, but it will error out because `__method_a` is not defined at that exact level, even though it is defined for one level up and one level down.

In [23]:
%reset -f

In [24]:
class Root:
    def __method_a(self):
        print("Root::__method_a")


class LevelOne(Root):
    def method_b(self):
        print("LevelOne::method_b")
        self.__method_a()


class LevelTwo(LevelOne):
    def __method_a(self):
        print("LevelTwo::__method_a")

In [25]:
ltwo = LevelTwo()
try:
    ltwo.method_b()
except AttributeError as ae:
    print(f"ERROR: {ae}")

LevelOne::method_b
ERROR: 'LevelTwo' object has no attribute '_LevelOne__method_a'


### Using `super()`

Python's built-in super function will look for the called method in MRO, but only in the object's
parents, not in the object itself.

In [26]:
%reset -f

In [27]:
class Root:
    def method_a(self):
        print("Root::method_a")


class LevelOne(Root):
    def method_a(self):
        print("LevelOne::method_a")

    def method_b(self):
        print("LevelOne::method_b")

        # This will inovke Root.method_a
        super().method_a()

In [29]:
lone = LevelOne()
lone.method_b()

LevelOne::method_b
Root::method_a


super works in the multiple inheritance case as expected. In the example below it invokes LevelOneB
when called inside LevelOneA because the MRO is LevelOneA, LevelOneB, and Root.

In [30]:
%reset -f

In [32]:
class Root:
    def method_a(self):
        print("Root::method_a")


class LevelOneA(Root):
    def method_a(self):
        print("LevelOneA::method_a")
        super().method_a()


class LevelOneB(Root):
    def method_a(self):
        print("LevelOneB::method_a")
        super().method_a()


class LevelTwo(LevelOneA, LevelOneB):
    def method_a(self):
        print("LevelTwo::method_a")
        super().method_a()

In [33]:
ltwo = LevelTwo()
ltwo.method_a()

LevelTwo::method_a
LevelOneA::method_a
LevelOneB::method_a
Root::method_a


Example with a different MRO, notice that LevelOneB is before LevelOneA.

In [ ]:
class SecondLevel(LevelOneB, LevelOneA):
    def method_a(self):
        print("SecondLevel::method_a")
        super().method_a()

In [36]:
l2 = SecondLevel()
l2.method_a()

SecondLevel::method_a
LevelOneB::method_a
LevelOneA::method_a
Root::method_a


## Object Construction

Python follows a 2 phase object construction, in the first phase it actually constructs the object
and in the second phase it initializes it.

In the first phase - when Python detects that a class is being called, it looks for the
static `__new__` method and does go up the object hierarchy even though the name starts with double
underscore. Eventually it will reach `object.__new__` which will instantiate a concrete object of
the right type.

In the second phase - Python will call `type.__call__` (see callable classes) which in turn will call
the current class's `__init__` method with all the args, kwargs that are needed. This also follows
normal rules of polymorphism inspite of having a name that starts with a double underscore. Which
means it will follow the MRO to invoke the first `__init__` it finds and calls to `super` will also
follow the MRO rules.

Below is a demo with a polymorphic `__init__`

In [37]:
%reset -f

In [40]:
class Root:
    def __init__(self):
        print("Root::__init__")


class LevelOneA(Root):
    def __init__(self):
        print("LevelOneA::__init__")
        super().__init__()


class LevelOneB(Root):
    def __init__(self):
        print("LevelOneB::__init__")
        super().__init__()


class LevelTwo(LevelOneA, LevelOneB):
    def __init__(self):
        print("LevelTwo::__init__")

        # MRO: LevelOneA.__init__ > LevelOneB.__init__ > Root.__init__
        super().__init__()

In [41]:
LevelTwo()

LevelTwo::__init__
LevelOneA::__init__
LevelOneB::__init__
Root::__init__


Of course if the calls to super had been missing, it would stop at the first `__init__` it finds. Unlike
the constructor semantics in other OO langs, the call is not automatically propagated up the object
hierarchy.

In [42]:
%reset -f

In [43]:
class Root:
    def __init__(self):
        print("Root::__init__")


class LevelOneA(Root):
    def __init__(self):
        print("LevelOneA::__init__")


class LevelOneB(Root):
    def __init__(self):
        print("LevelOneB::__init__")


class LevelTwo(LevelOneA, LevelOneB):
    def __init__(self):
        print("LevelTwo::__init__")
        super().__init__()  # Will stop at LevelOneA.__init__

In [44]:
LevelTwo()

LevelTwo::__init__
LevelOneA::__init__


### Demo with `__new__`

In [46]:
class MyDemo:
    def __new__(cls, *args, **kwargs):
        print("MyDemo.__new__")
        return super().__new__(cls)

In [47]:
MyDemo()

MyDemo.__new__


The thing with `__new__` is that it can return anything it wants. It does not *have* to return an instance
of the type. Typically, calling `super().__new__` at each level will propagate the call all the way to
the mother `object.__new__` which knows how to instantiate the object. Another wierdness about `__new__`
is that it is a static method. It does not have to be decorated as such.

In [48]:
class MyClass:
    def __new__(cls, *args, **kwargs):
        print("MyClass.__new__")
        return 1

In [49]:
obj = MyClass()

MyClass.__new__


In [50]:
type(obj)

int

Demo that `object.__new__` instantiates the object and calls `__init__`
One implementation issue to note here is that if I am overriding both `__new__` and `__init__` then I
should not call `super().__new__(cls, *args, **kwargs)` in the base class that is implicitly derived
from `object`. In this case I only need to call `super().__new__(cls)`. And `type.__call__` will do the
rest of the magic. For any other base class that is not implicitly deriving from object, I'll still
need to make the full `super().__new__(cls, *args, **kwargs)` call.

In [51]:
class MyDemo:
    def __new__(cls, *args, **kwargs):
        print("MyDemo.__new__")
        return super().__new__(cls)

In [52]:
MyDemo()

MyDemo.__new__


In [53]:
class MyAwesomeDemo(MyDemo):
    def __new__(cls, *args, **kwargs):
        print("MyAwesomeDemo.__new__")
        return super().__new__(cls, *args, **kwargs)

    def __init__(self, name, duration):
        print(f"MyAwesomeDemo {name} is for {duration} minutes")

In [54]:
MyAwesomeDemo("pitch", 10)

MyAwesomeDemo.__new__
MyDemo.__new__
MyAwesomeDemo pitch is for 10 minutes


For multilevel inheritance, `__new__` is called based on MRO. But just like `__init__`, it will stop at
the first `__new__` impl it finds. For this reason, it is the responsibility of each `__new__` impl to
call `super().__new__` at the appropriate time. Otherwise the call will not propagate up.